### **Stage 3: DeepAgent**

##### **Importing Libraries**

In [39]:
import os
import time
import inspect
from typing import Literal
from pydantic import BaseModel, Field
from deepagents import create_deep_agent

##### **Model**

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("telecom_assg2")

if not api_key:
    raise RuntimeError("Set telecom_assg2 in your .env file before running the agent cells.")

model_name = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

model = ChatGroq(model= model_name, temperature= 0, api_key= api_key, model_kwargs={"tool_choice": "auto"},)

##### **Mock Telecom Database**

In [41]:
customer_details = {
    "AC10234": {"name": "L. Kim",       "phone_number": "212-555-0142", "plan": "unlimited_plus", "area_code": "212", "monthly_bill_usd": 85,  "data_used_gb": 42},
    "AC20458": {"name": "D. Osei",      "phone_number": "310-555-0198", "plan": "standard_20gb",  "area_code": "310", "monthly_bill_usd": 45,  "data_used_gb": 12},
    "AC30671": {"name": "P. Novak",     "phone_number": "404-555-0113", "plan": "basic_5gb",      "area_code": "404", "monthly_bill_usd": 30,  "data_used_gb": 4.5},
    "AC40892": {"name": "R. Alvarez",   "phone_number": "512-555-0176", "plan": "business_50gb",  "area_code": "512", "monthly_bill_usd": 120, "data_used_gb": 38},
    "AC50103": {"name": "S. Chen",      "phone_number": "606-555-0159", "plan": "standard_20gb",  "area_code": "606", "monthly_bill_usd": 45,  "data_used_gb": 15},
    "AC60217": {"name": "T. Barros",    "phone_number": "213-555-0134", "plan": "basic_5gb",      "area_code": "213", "monthly_bill_usd": 30,  "data_used_gb": 6.2},
    "AC70345": {"name": "N. Whitfield", "phone_number": "718-555-0187", "plan": "unlimited_plus", "area_code": "718", "monthly_bill_usd": 85,  "data_used_gb": 55},
    "AC80456": {"name": "M. Kowalski",  "phone_number": "702-555-0121", "plan": "business_50gb",  "area_code": "702", "monthly_bill_usd": 120, "data_used_gb": 21}
}

plan_details = {
    "basic_5gb":      {"data_gb": 5,           "minutes": "unlimited", "price_usd": 30,  "overage_fee_per_gb": 10},
    "standard_20gb":  {"data_gb": 20,          "minutes": "unlimited", "price_usd": 45,  "overage_fee_per_gb": 8},
    "unlimited_plus": {"data_gb": "unlimited", "minutes": "unlimited", "price_usd": 85,  "overage_fee_per_gb": 0},
    "business_50gb":  {"data_gb": 50,          "minutes": "unlimited", "price_usd": 120, "overage_fee_per_gb": 6},
    "family_100gb":   {"data_gb": 100,         "minutes": "unlimited", "price_usd": 150, "overage_fee_per_gb": 5}
}

network_status = {
    "212": {"status": "Outage",      "outage_hours": 18, "affected_services": ["voice", "data"],       "region": "New York, NY",    "technician_dispatched": True},
    "310": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Los Angeles, CA", "technician_dispatched": False},
    "404": {"status": "Degraded",    "outage_hours": 3,  "affected_services": ["data"],                "region": "Atlanta, GA",     "technician_dispatched": False},
    "512": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Austin, TX",      "technician_dispatched": False},
    "606": {"status": "Outage",      "outage_hours": 30, "affected_services": ["voice", "data", "sms"],"region": "Lexington, KY",   "technician_dispatched": True},
    "213": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Los Angeles, CA", "technician_dispatched": False},
    "718": {"status": "Degraded",    "outage_hours": 5,  "affected_services": ["voice"],               "region": "Brooklyn, NY",    "technician_dispatched": True},
    "702": {"status": "Operational", "outage_hours": 0,  "affected_services": [],                      "region": "Las Vegas, NV",   "technician_dispatched": False}
}

##### **Defining Tools**

In [42]:
# TOOL 1
def lookup_account(customer_id: str):
    """Look up an account by its account_id, e.g., 'AC10234'"""
    
    customer_info = customer_details.get(customer_id.upper())
    
    # If customer information not found, return an error message
    if not customer_info:
        return {"error": f"No information found for {customer_id}."}    
    return customer_info
    
# TOOL 2
def check_network_status(area_code: str | int):
    """Check the current network status for a 3-digit area code, e.g. '212'."""
    
    area_code = str(area_code).strip()

    # Look up the network status.
    status = network_status.get(area_code)
    
    # If area code is not recognized, return an error message
    if not status:
        return {"error": f"No network status found for area code '{area_code}"}
    return status    

# TOOL 3
def request_plan_change(customer_id: str, new_plan: str):
    """Check whether a requested plan exists and calculate the monthly price difference. 
    Does NOT change the plan — only quotes it."""

    account = lookup_account(customer_id)
    
    # Check whether the customer exists
    if "error" in account:
        return {"error": f"Unknown customer ID '{customer_id}'"}
    
    new_plan = new_plan.lower()
    target = plan_details.get(new_plan)
    
    # Check whether the requested plan exists, If target is None, the requested plan does not exist.
    if not target:
        return {"available": False, "reason": f"Unknown plan '{new_plan}'. Valid: basic_5gb, standard_20gb, unlimited_plus, business_50gb."}
    
    current = plan_details[account["plan"]]
    
    # Calculate the price difference
    price_diff = round(target["price_usd"] - current["price_usd"], 2)
    
    # Return the plan-change quotation
    return {"available": True, "current_plan": account["plan"], "new_plan": new_plan, "price_diff_usd": price_diff}
  

##### **AGENTS.md — House Rules**

In [43]:
AGENTS_MD = """\
# TeleAssist: Telecom Customer Support — Agent Instructions

## Rules
- Never process or approve a billing credit without explicit customer confirmation of the amount and reason.
- Always state the network status verbatim from check_network_status — never estimate or infer outage duration.
- If a request spans more than one topic (e.g. network status + billing credit), address every part
  before finishing; do not silently drop part of a multi-part request.
"""

with open("AGENTS.md", "w") as f:
    f.write(AGENTS_MD)

print("AGENTS.md written.")


AGENTS.md written.


##### **Structured Output Schema**

In [44]:
class AgentResponse(BaseModel):
    response: str = Field(description= "The agent's response to the user query.")
    category: Literal["network_status", "plan_details", "account", "plan_change", "other"] = Field(description= "What the query was about.")
    summary: str = Field(description= "A short summary of what was looked up. Empty string if nothing was looked up.")

##### **Sub-agents**

In [45]:
# Subagent 1
network_status_subagent = {
    "name": "network_status_agent",
    "description": "Handles network outage and network status questions.",
    "system_prompt": """
You are the network status specialist.

If a customer ID is provided:
1. Use lookup_account to find the account.
2. Get the area_code from the account.
3. Use check_network_status with that area code.
4. Report the actual result.

Never use a customer ID as an area code.
Do not invent information.
""",
    "tools": [lookup_account, check_network_status]}

# Subagent 2
plan_change_subagent = {
    "name": "plan_change_agent",
    "description": "Handles telecom plan change requests and pricing.",
    "system_prompt": """
You are the plan change specialist.

Use request_plan_change to calculate the price difference
between the customer's current plan and requested plan.

Report the current plan, requested plan, and monthly
price difference.

The tool only provides a quote.
It does not actually change the customer's plan.

Do not invent information.
""",
    "tools": [lookup_account, request_plan_change]}

subagents = [network_status_subagent, plan_change_subagent]

In [46]:
coordinator_prompt = (
    "You are TeleAssist. Route every request to exactly one specialist by calling task():\n"
    "- network_status_agent: outages, network issues\n"
    "- plan_change_agent: plan changes\n"
    "Delegate immediately, do not answer directly. Follow the house rules in AGENTS.md. "
    "Always return the final answer in the required structured format."
)

_create_kwargs = dict(
    model= model,
    tools= [],
    subagents= subagents,
    system_prompt= coordinator_prompt,
    response_format= AgentResponse,
)

In [47]:
result = stage3_agent.invoke({
    "messages": [{"role": "user", "content": ("I have account AC10234. What would it cost to change my plan to business_50gb?")}]
})

print(result["structured_response"])

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=task({"subagent_type": "plan_change_agent", "description": "Provide the cost to change plan to business_50gb for account AC10234"})</function>'}}

#### **Writeup: What changed between stages, and why the old code couldn't just be kept**

**Stage 1 -> Stage 2**

Stage 1 was single `create_agent` with **3** tools `lookup_account`, `check_network_status` and `request_plan_change`, **2** `PIIMiddleware` and a Pydantic response via `response_format`. It was entirely upto the LLM, which tools to call, in what order and how to combine the results. stage 2 rebulit the same task as an explicit graph state, deterministic routing, a pause point for human approval, and cross-turn memory — none of which a single agent call can express. So the LLM-driven routing was replaced with a hand-coded `supervisor_node` (keyword rules, not an LLM call), the tool functions were called directly from node functions instead of via LLM tool-calls, `plan_change_node` added `interrupt()`/`Command(resume=...)` for approval, and an `InMemorySaver` checkpointer was added for thread memory. 

**What broke:** 

customer IDs were briefly being treated as area codes, and `account` was referenced before it was initialized — both from state/order assumptions that didn't exist in Stage 1's single-call flow. Fixed by forcing `lookup_account` to run first and passing its `area_code` field forward instead of reusing raw user input.

**Why Stage 1's code couldn't just be kept:**

Stage 1 has no notion of graph state, routing, or a pause/resume point — the LLM call is a single opaque step. None of that maps onto a `StateGraph`; conditional edges, node functions, and a checkpointer had to be built from scratch, and the routing logic had to be made explicit instead of implicit in a system prompt.

**Stage 2 -> Stage 3**

Stage 2's supervisor and routing edges are a hand-built version
of exactly what `create_deep_agent` does internally via `task()` delegation. Keeping both would mean two competing orchestration layers, so the graph was dropped: the coordinator got `tools=[]` and now only delegates to declared sub-agents (`network_status_agent`, `plan_change_agent`), and global rules that lived in the Stage 1 prompt / Stage 2 node logic moved into an `AGENTS.md` house-rules file, since that's the convention the framework expects rather than something wired by hand.

**What broke:** 

Direct tool calls worked fine on Groq, but the framework's internal `task()` call — used for delegating to a sub-agent — triggered a `tool_use_failed` error even though the model generated a correct `subagent_type`. That's a provider/framework tool-schema incompatibility— it required moving to the sub-agent/`task()` pattern itself.

**Why Stage 2's graph couldn't just be kept:** 

Stage 2's routing (`supervisor_node`, `route_from_supervisor`, explicit edges) is a hand-built alternative to exactly what `create_deep_agent` provides internally — task delegation, sub-agent selection, and orchestration are handled by the framework's own `task()` tool. Keeping the stage 2 graph and adding sub-agents on top would have meant running two competing orchestration layers; the graph had to be dropped in favor of describing sub-agents declaratively (`name`, `description`, `system_prompt`, `tools`) and letting the coordinator's `task()` calls do the routing that the supervisor node used to do.